In [ ]:
scp VMware-VCSA-all-7.0.3-24927011.iso control-plane-3:/opt/

In [ ]:
# # Check if ovftool can run at all
# /mnt/vcsa/vcsa/ovftool/lin64/ovftool --version

# # Or check for missing libraries
# ldd /mnt/vcsa/vcsa/ovftool/lin64/ovftool

In [ ]:
# Install libnsl (provides libnsl.so.1)
yum install -y libnsl

# Or if that doesn't work, try the compatibility package
yum install -y libnsl2

In [ ]:
# # Find where libnsl is installed
# find /usr/lib* -name "libnsl*.so*" 2>/dev/null

# # If you have libnsl.so.2 but not .so.1, create symlink
# ln -s /usr/lib64/libnsl.so.2 /usr/lib64/libnsl.so.1

# # Or for 32-bit if needed
# ln -s /usr/lib/libnsl.so.2 /usr/lib/libnsl.so.1

In [ ]:
ls -la /mnt/vcsa/vcsa/ovftool/lin64/ovftool

# If not executable, make it so:
chmod +x /mnt/vcsa/vcsa/ovftool/lin64/ovftool

In [ ]:
sudo mkdir -p /mnt/vcsa
sudo mount -o loop /opt/VMware-VCSA-all-7.0.3-24927011.iso /mnt/vcsa

In [ ]:
cp /mnt/vcsa/vcsa-cli-installer/templates/install/embedded_vCSA_on_ESXi.json /root/vcsa-deploy.json

In [ ]:
cat > vcsa-deploy.json << EOF
{
    "__version": "2.13.0",
    "__comments": "VCSA 7.0.3 deployment on ESXi 172.20.0.151",
    "new_vcsa": {
        "esxi": {
            "hostname": "172.20.0.151",
            "username": "root",
            "password": "root@123",
            "deployment_network": "VM Network",
            "datastore": "datastore1"
        },
        "appliance": {
            "thin_disk_mode": true,
            "deployment_option": "small",
            "name": "vcsa-01"
        },
        "network": {
            "ip_family": "ipv4",
            "mode": "static",
            "system_name": "172.20.0.101",
            "ip": "172.20.0.101",
            "prefix": "16",
            "gateway": "172.20.0.254",
            "dns_servers": [
                "172.16.1.16"
            ]
        },
        "os": {
            "password": "Root@123",
            "ntp_servers": "pool.ntp.org",
            "ssh_enable": true
        },
        "sso": {
            "password": "Root@123",
            "domain_name": "vsphere.local"
        }
    },
    "ceip": {
        "settings": {
            "ceip_enabled": false
        }
    }
}
EOF

In [ ]:
cd /mnt/vcsa/vcsa-cli-installer/lin64

In [ ]:
# 1. Verify template (no EULA needed - already passed)
./vcsa-deploy install --verify-template-only /root/vcsa-deploy.json

# 2. Precheck - MUST include --accept-eula
./vcsa-deploy install --accept-eula --precheck-only /root/vcsa-deploy.json

# 3. Actual deployment - MUST include --accept-eula
./vcsa-deploy install \
    --accept-eula \
    --no-ssl-certificate-verification \
    --no-esx-ssl-verify \
    /root/vcsa-deploy.json